# Are the remaining false alarms water? A check against Sentinel-2

The best detector of [notebook 04](04_false_alarms_s1.ipynb) still has a FAR of 0.74 on independent labels. Most of those pixels got darker between the two Sentinel-1 passes, on flat ground ([notebook 05](05_terrain_s1.ipynb)), far from the mapped flood. Either the detector is wrong, or the water is real and the EMS maps do not show it.

This notebook looks at EMSR664, which holds 111 of the 130 km² of independent labels, in a third image: the Sentinel-2 pass of **23 May 2023, 10:06 UTC**, about 29 hours after the Sentinel-1 pass, with 2–9 % cloud on its tiles. EMSR762, the next largest, is too cloudy on the nearby passes to test.

**The test is one-sided.** Water seen by Sentinel-2 confirms the radar. Water *not* seen proves little: it may have receded in those 29 hours, or lie under vegetation. So every number is compared with what Sentinel-2 sees under the mapped flood itself, under ordinary dry land, and under surfaces dark in both radar passes.

Cost: about **20 PU** once (five images at 40 m, three 16-bit bands), checked against a budget before anything is sent. The radar images come from notebook 02.

In [ ]:
import csv
import math
import os
from io import BytesIO
from pathlib import Path

import numpy as np

ACTIVATIONS = [
    "EMSR659",
    "EMSR664",
    "EMSR705",
    "EMSR762",
    "EMSR795",
    "EMSR800",
    "EMSR843",
    "EMSR855",
]
TEST = "EMSR664"
RESOLUTION_M = 40
CHIP = 128
SEED = 0
BUDGET_PU = float(os.environ.get("NOTEBOOK_BUDGET_PU", 25))  # nothing is sent above it
OPTICAL_FROM, OPTICAL_TO = "2023-05-23T09:50:00Z", "2023-05-23T10:20:00Z"

DATA_DIR = Path(os.environ.get("COPERNICUS_EMS_DATA_DIR", "~/.cache/copernicus-ems")).expanduser()
DATASET = DATA_DIR / "datasets" / f"italy-floods-s1-change-{RESOLUTION_M}m"
OPTICAL_DIR = DATA_DIR / TEST / "sentinel-2"
FIGURE = Path(os.environ.get("NOTEBOOK_FIGURE", "../docs/images/notebook-optical-s2.png"))
print("data:", DATA_DIR, "| dataset:", DATASET)

## 1. The five-band dataset of notebook 02

The dataset is rebuilt from the pairs in the data directory, locally and for free. Its train/val/test column is ignored: the folds are made by activation below.

In [ ]:
from copernicus_ems.dataset import build_dataset, find_pairs

pairs = find_pairs(DATA_DIR, ACTIVATIONS)
assert any(p.reference is not None for p in pairs), (
    "no pair with a reference: run notebook 02 first"
)
summary = build_dataset(pairs, DATASET, size=CHIP, min_observed=0.5, seed=SEED, reference=True)
rows = list(csv.DictReader((DATASET / "index.csv").open()))
made_from = {
    (p.activation, p.product): sorted({i["sensor"] for i in p.meta.get("made_from", [])})
    for p in pairs
}
for r in rows:
    r["made_from"] = ", ".join(made_from.get((r["activation"], r["product"]), []))
    r["same_image"] = "Sentinel-1" in r["made_from"]
FOLDS = sorted({r["activation"] for r in rows})
print(
    f"{summary.pairs} pairs -> {summary.chips} chips of {CHIP} px; {len(FOLDS)} folds: {', '.join(FOLDS)}"
)

## 2. Features

The eleven features of notebook 02: VV and VH in dB, their ratio, local mean and spread of VV, the pre-event VV and VH, and the changes ΔVV, ΔVH, raw and smoothed over 5×5 pixels.

In [ ]:
import rasterio
from scipy.ndimage import uniform_filter

from copernicus_ems.ml import FLOODED, IGNORE, LabelRaster

SINGLE = slice(0, 5)
NAMES = [
    "VV",
    "VH",
    "VV-VH",
    "VV mean",
    "VV spread",
    "VV pre",
    "VH pre",
    "dVV",
    "dVH",
    "dVV mean",
    "dVH mean",
]


def db(x):
    return 10 * np.log10(np.maximum(x, 1e-6))


def features(path):
    with rasterio.open(path) as src:
        vv, vh, vv_pre, vh_pre, _mask = src.read()
    vv_db, vh_db, vvp_db, vhp_db = db(vv), db(vh), db(vv_pre), db(vh_pre)
    mean = uniform_filter(vv_db, 5)
    spread = np.sqrt(np.maximum(uniform_filter(vv_db**2, 5) - mean**2, 0))
    dvv, dvh = vv_db - vvp_db, vh_db - vhp_db
    return np.stack(
        [
            vv_db,
            vh_db,
            vv_db - vh_db,
            mean,
            spread,
            vvp_db,
            vhp_db,
            dvv,
            dvh,
            uniform_filter(dvv, 5),
            uniform_filter(dvh, 5),
        ],
        axis=-1,
    ).astype(np.float32)


X = np.stack([features(DATASET / r["image"]) for r in rows])
LABELS = np.stack([LabelRaster.from_geotiff(DATASET / r["label"]).labels for r in rows])
Y = [LabelRaster.from_geotiff(DATASET / r["label"]) for r in rows]
km2 = (RESOLUTION_M / 1000) ** 2
print(
    f"{'activation':10} {'chips':>5} {'flooded km²':>12} {'independent km²':>16} {'same image km²':>15}"
)
for a in FOLDS:
    idx = [i for i, r in enumerate(rows) if r["activation"] == a]
    wet = [int((LABELS[i] == FLOODED).sum()) for i in idx]
    ind = sum(w for i, w in zip(idx, wet, strict=True) if not rows[i]["same_image"])
    print(
        f"{a:10} {len(idx):5d} {sum(wet) * km2:12.1f} {ind * km2:16.1f} {(sum(wet) - ind) * km2:15.1f}"
    )

## 3. The detector, tuned without EMSR664

The rule of notebook 04, VV < *t* dB, VV_pre ≥ *t*ₚ dB, patches of at least *n* pixels, with all three chosen on the other activations only.

In [ ]:
from scipy.ndimage import label as connected

from copernicus_ems.ml import Scores, evaluate

OBSERVED = LABELS != IGNORE
WET = LABELS == FLOODED
EIGHT = np.ones((3, 3), dtype=bool)
VV, PRE = X[..., 0], X[..., 5]


def csi(chips, predicted):
    hits = int((predicted & WET[chips]).sum())
    false = int((predicted & OBSERVED[chips] & ~WET[chips]).sum())
    return hits / max(int(WET[chips].sum()) + false, 1)


def min_size(predicted, n):
    if n <= 1:
        return predicted
    labelled, _ = connected(predicted, structure=EIGHT)
    big = np.bincount(labelled.ravel()) >= n
    big[0] = False
    return big[labelled]


train = np.flatnonzero(np.array([r["activation"] != TEST for r in rows]))
test = np.flatnonzero(np.array([r["activation"] == TEST for r in rows]))
t, tp = max(
    [(a, b) for a in np.arange(-22, -10, 0.5) for b in np.arange(-22, -8, 1.0)],
    key=lambda c: csi(train, (VV[train] < c[0]) & (PRE[train] >= c[1])),
)
base = (VV[train] < t) & (PRE[train] >= tp)
n = max((1, 2, 4, 8, 16, 32), key=lambda k: csi(train, np.stack([min_size(b, k) for b in base])))
PREDICTED = np.zeros_like(WET)
for i in test:
    PREDICTED[i] = min_size((VV[i] < t) & (PRE[i] >= tp), n)
scores = sum((evaluate(PREDICTED[i], Y[i]) for i in test), Scores.zero())
print(f"VV < {t:.1f} dB, VV_pre >= {tp:.0f} dB, patches >= {n} px")
print(
    f"{TEST}: CSI {scores.csi:.3f}  POD {scores.pod:.3f}  FAR {scores.far:.3f}  bias {scores.bias:.2f}"
)

## 4. The Sentinel-2 pass, on the grids of the radar images

Green (B03), near infrared (B08) and the scene classification (SCL) on exactly the grid of each radar image, as 16-bit integers. The cost is computed and checked before the first request; images already in the data directory cost nothing.

In [ ]:
from copernicus_ems.sentinelhub import SentinelHub, estimate_pu

EVALSCRIPT = """//VERSION=3
function setup() {
  return {input: ["B03", "B08", "SCL", "dataMask"],
          output: {id: "default", bands: 4, sampleType: "UINT16"}};
}
function evaluatePixel(s) { return [s.B03 * 10000, s.B08 * 10000, s.SCL, s.dataMask]; }
"""

radar = {}  # radar image -> its pairs
used = {r["product"] for r in rows if r["activation"] == TEST}  # only pairs with chips
for p in pairs:
    if p.activation == TEST and p.product in used:
        radar.setdefault(p.image, []).append(p)


def optical_path(image):
    return OPTICAL_DIR / f"S2_20230523T1006_{image.stem}.tif"


def optical_request(src):
    left, bottom, right, top = src.bounds
    return {
        "input": {
            "bounds": {
                "bbox": [left, bottom, right, top],
                "properties": {"crs": f"http://www.opengis.net/def/crs/EPSG/0/{src.crs.to_epsg()}"},
            },
            "data": [
                {
                    "type": "sentinel-2-l2a",
                    "dataFilter": {"timeRange": {"from": OPTICAL_FROM, "to": OPTICAL_TO}},
                }
            ],
        },
        "output": {
            "width": src.width,
            "height": src.height,
            "responses": [{"identifier": "default", "format": {"type": "image/tiff"}}],
        },
        "evalscript": EVALSCRIPT,
    }


work = []
for image in radar:
    with rasterio.open(image) as src:
        pu = estimate_pu(src.width, src.height, bands=3, float32=False, orthorectify=False)
        if not optical_path(image).is_file():
            work.append((image, optical_request(src), pu, src.profile))
cost = sum(w[2] for w in work)
print(
    f"{len(radar)} radar images, {len(work)} optical images to fetch, {cost:.1f} PU (budget {BUDGET_PU:.0f})"
)
assert cost <= BUDGET_PU, f"{cost:.1f} PU is over the budget of {BUDGET_PU}"

if work:
    OPTICAL_DIR.mkdir(parents=True, exist_ok=True)
    async with SentinelHub() as hub:
        for image, request, pu, profile in work:
            body = await hub.process(request, pu=pu)
            with rasterio.open(BytesIO(body)) as got:
                data = got.read()
            assert data.shape == (4, profile["height"], profile["width"]), data.shape
            profile = {**profile, "count": 4, "dtype": "uint16", "nodata": None}
            with rasterio.open(optical_path(image), "w", **profile) as dst:
                dst.write(data)
    print(f"spent about {hub.pu_estimated:.1f} PU")

## 5. Whole images, and every map of an area at once

The chips above repeat a pixel when an area has two maps: AOI01 is mapped by `DEL_MONIT03` and `GRA_PRODUCT` on one radar image. From here on each radar image counts once. A pixel is **mapped flood** if *any* map of that image says so, **dry** if at least one map observed it and none says flooded.

A pixel is **clear** where Sentinel-2 has data and SCL does not say cloud, cloud shadow, cirrus, snow or saturated. Among clear pixels, **water** is NDWI = (green − NIR) / (green + NIR) above a threshold, or SCL class 6.

In [ ]:
NOT_CLEAR = (0, 1, 3, 8, 9, 10, 11)
images = []
for image, ps in radar.items():
    with rasterio.open(image) as src:
        vv = db(src.read(1))
        transform, crs, bounds = src.transform, src.crs, src.bounds
    with rasterio.open(ps[0].reference) as src:
        pre = db(src.read(1))
    with rasterio.open(optical_path(image)) as src:
        green, nir, scl, has = src.read().astype(np.float32)
    maps = {}
    for p in ps:
        with rasterio.open(p.labels) as src:
            maps[p.product] = src.read(1)
    images.append(
        {
            "image": image,
            "transform": transform,
            "crs": crs,
            "bounds": bounds,
            "vv": vv,
            "pre": pre,
            "predicted": min_size((vv < t) & (pre >= tp), n),
            "maps": maps,
            "wet": np.any([m == FLOODED for m in maps.values()], axis=0),
            "observed": np.any([m != IGNORE for m in maps.values()], axis=0),
            "clear": (has > 0) & ~np.isin(scl, NOT_CLEAR),
            "ndwi": (green - nir) / np.maximum(green + nir, 1),
            "scl": scl,
        }
    )
observed_px = sum(int(d["observed"].sum()) for d in images)
clear_px = sum(int((d["observed"] & d["clear"]).sum()) for d in images)
print(
    f"{len(images)} images, {observed_px * km2:.0f} km² observed, clear on {clear_px / observed_px:.2%}"
)

## 6. Two EMS maps of one area, one day apart

The same detector, on the same radar image, scored against each map and against both.

In [ ]:
def counts(d, wet, observed):
    predicted = d["predicted"]
    return np.array(
        [(predicted & wet).sum(), (~predicted & wet).sum(), (predicted & observed & ~wet).sum()],
        dtype=np.int64,
    )


def line(name, c):
    hits, misses, false = c
    print(
        f"{name:34} {(hits + misses) * km2:8.1f} {false * km2:9.1f} {hits / max(hits + misses + false, 1):7.3f}"
    )


print(f"{'':34} {'mapped km²':>8} {'false km²':>9} {'CSI':>7}")
each, union = np.zeros(3, dtype=np.int64), np.zeros(3, dtype=np.int64)
for d in images:
    for product, m in d["maps"].items():
        c = counts(d, m == FLOODED, m != IGNORE)
        each += c
        line(product, c)
    union += counts(d, d["wet"], d["observed"])
    if len(d["maps"]) > 1:
        a, b = (m == FLOODED for m in d["maps"].values())
        print(
            f"  the two maps share {(a & b).sum() * km2:.1f} km²; {(a | b).sum() * km2:.1f} km² in either"
        )
print()
line("every map apart", each)
line("any map of the image", union)

## 7. What Sentinel-2 sees under each group of pixels

* **dark in both radar passes**: a control. It holds permanent water, but also shadow and tarmac.
* **mapped flood, found / missed**: the EMS polygons, split by the radar detector. The first row says how much water survives to the Sentinel-2 pass under a confirmed flood.
* **false alarms**: predicted by the radar detector, in no map. This is the question.
* **quiet dry land**: in no map, not predicted, a control for what NDWI marks with no flood.

Shares are of the clear pixels of each group.

In [ ]:
def group_masks(d):
    wet, dry = d["wet"], d["observed"] & ~d["wet"]
    dark = (d["vv"] < t) & (d["pre"] < tp)
    return {
        "dark in both radar passes": dry & dark,
        "mapped flood, found": wet & d["predicted"],
        "mapped flood, missed": wet & ~d["predicted"],
        "false alarms": dry & d["predicted"],
        "quiet dry land": dry & ~d["predicted"] & ~dark,
    }


GROUP_NAMES = [
    "dark in both radar passes",
    "mapped flood, found",
    "mapped flood, missed",
    "false alarms",
    "quiet dry land",
]
overall = {k: dict.fromkeys(["px", "clear", "ndwi0", "ndwi2", "scl"], 0) for k in GROUP_NAMES}
for d in images:
    for name, mask in group_masks(d).items():
        clear = mask & d["clear"]
        r = overall[name]
        r["px"] += int(mask.sum())
        r["clear"] += int(clear.sum())
        r["ndwi0"] += int((clear & (d["ndwi"] > 0)).sum())
        r["ndwi2"] += int((clear & (d["ndwi"] > 0.2)).sum())
        r["scl"] += int((clear & (d["scl"] == 6)).sum())
print(f"{'':28} {'km²':>7} {'clear':>7} {'NDWI>0':>8} {'NDWI>0.2':>9} {'SCL water':>10}")
for name, r in overall.items():
    c = max(r["clear"], 1)
    print(
        f"{name:28} {r['px'] * km2:7.1f} {r['clear'] / max(r['px'], 1):7.0%} {r['ndwi0'] / c:8.1%} {r['ndwi2'] / c:9.1%} {r['scl'] / c:10.1%}"
    )

## 8. Which water? The land cover under the false alarms

Water in a Sentinel-2 image is not always flood: a lagoon whose surface was rough in the reference pass and calm in the flood pass also "gets darker", and it is water in the optical image too. [ESA WorldCover 2021](https://esa-worldcover.org) (public, 0 PU) separates permanent water and wetland from cropland. It is read once for each image and kept in the data directory.

It **cannot** separate flood from irrigation: a rice field flooded on purpose in May is cropland with open water, and rice is grown in parts of the Po plain.

In [ ]:
from rasterio.enums import Resampling
from rasterio.warp import reproject, transform_bounds

WORLDCOVER = "https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map"
COVER = {
    10: "tree",
    20: "shrub",
    30: "grass",
    40: "crop",
    50: "built",
    60: "bare",
    80: "water",
    90: "wetland",
}
COVER_DIR = DATA_DIR / TEST / "worldcover"


def worldcover(d):
    path = COVER_DIR / f"{d['image'].stem}.tif"
    if path.is_file():
        with rasterio.open(path) as src:
            return src.read(1)
    height, width = d["vv"].shape
    west, south, east, north = transform_bounds(d["crs"], "EPSG:4326", *d["bounds"])
    cover = np.zeros((height, width), np.uint8)
    for lat in range(math.floor(south / 3) * 3, math.floor(north / 3) * 3 + 1, 3):
        for lon in range(math.floor(west / 3) * 3, math.floor(east / 3) * 3 + 1, 3):
            name = f"ESA_WorldCover_10m_2021_v200_N{lat:02d}E{lon:03d}_Map.tif"
            part = np.zeros((height, width), np.uint8)
            with rasterio.open(f"/vsicurl/{WORLDCOVER}/{name}") as src:
                reproject(
                    rasterio.band(src, 1),
                    part,
                    dst_transform=d["transform"],
                    dst_crs=d["crs"],
                    resampling=Resampling.mode,
                    dst_nodata=0,
                )
            cover = np.where(cover == 0, part, cover)
    COVER_DIR.mkdir(parents=True, exist_ok=True)
    with rasterio.open(
        path,
        "w",
        driver="GTiff",
        width=width,
        height=height,
        count=1,
        dtype="uint8",
        crs=d["crs"],
        transform=d["transform"],
    ) as dst:
        dst.write(cover, 1)
    return cover


with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR"):
    for d in images:
        d["cover"] = worldcover(d)

rows_cover = {}
for d in images:
    masks = group_masks(d)
    water = d["clear"] & (d["ndwi"] > 0)
    for value, name in COVER.items():
        here = d["cover"] == value
        r = rows_cover.setdefault(name, [0, 0, 0, 0])
        r[0] += int((masks["false alarms"] & here).sum())
        r[1] += int((masks["false alarms"] & here & water).sum())
        wet = masks["mapped flood, found"] | masks["mapped flood, missed"]
        r[2] += int((wet & here).sum())
        r[3] += int((wet & here & water).sum())
total = sum(r[0] for r in rows_cover.values())
print(f"{'':9} {'false km²':>10} {'share':>7} {'S2 water':>9} | {'mapped km²':>11} {'S2 water':>9}")
for name, r in sorted(rows_cover.items(), key=lambda kv: -kv[1][0]):
    print(
        f"{name:9} {r[0] * km2:10.1f} {r[0] / max(total, 1):7.1%} {r[1] / max(r[0], 1):9.1%} | {r[2] * km2:11.1f} {r[3] / max(r[2], 1):9.1%}"
    )

## 9. How much could the score change?

Against every map apart, against any map of the image, then two **upper bounds** that call a false alarm a hit if Sentinel-2 sees water there: all of them, or only outside permanent water and wetland. A hit does not change the union of predicted and mapped water, so CSI is (hits + confirmed) / (hits + misses + false alarms). These are bounds under an assumption, not measured scores: they would count an irrigated field as flood.

In [ ]:
hits, misses, false = union
denominator = hits + misses + false
confirmed = {"all": 0, "land": 0}
for d in images:
    candidate = group_masks(d)["false alarms"] & d["clear"] & (d["ndwi"] > 0)
    confirmed["all"] += int(candidate.sum())
    confirmed["land"] += int((candidate & ~np.isin(d["cover"], (80, 90))).sum())
print(f"every map apart                     CSI {each[0] / each.sum():.3f}")
print(f"any map of the image                CSI {hits / denominator:.3f}")
print(
    f"+ S2 water, outside water, wetland  CSI {(hits + confirmed['land']) / denominator:.3f}  ({confirmed['land'] * km2:.1f} km²)"
)
print(
    f"+ S2 water, everywhere              CSI {(hits + confirmed['all']) / denominator:.3f}  ({confirmed['all'] * km2:.1f} km²)"
)

## 10. Figure

Share of clear pixels that Sentinel-2 calls water (NDWI > 0), per group.

In [ ]:
import matplotlib.pyplot as plt

values = [overall[k]["ndwi0"] / max(overall[k]["clear"], 1) * 100 for k in GROUP_NAMES]
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.barh(GROUP_NAMES, values, height=0.6, color="#2a78d6")
for k, v in enumerate(values):
    ax.text(v + 1, k, f"{v:.1f} %", va="center", fontsize=9, color="#444")
ax.invert_yaxis()
ax.set_xlim(0, max(values) * 1.15 + 2)
ax.set_xlabel(f"clear pixels with NDWI > 0 on 23 May 2023, 10:06 UTC, {TEST} (%)")
ax.grid(axis="x", color="#e5e5e5")
ax.set_axisbelow(True)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
fig.text(
    0.5,
    0.005,
    "Contains modified Copernicus Sentinel data and Copernicus EMS information, © European Union",
    ha="center",
    fontsize=7,
)
fig.tight_layout(rect=(0, 0.03, 1, 1))
FIGURE.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE, dpi=110)
print("saved", FIGURE)

## Reading the scores

* **The labels are part of the low score.** Two EMS maps of AOI01, less than a day apart, share 0.5 km² of 52.1. Scored against either map of the image, instead of each apart, the same detector goes from CSI 0.16 to 0.23 with no optical data at all.
* **Most remaining false alarms are open water the next day.** 69 % of them are water for Sentinel-2, like 77 % of the mapped flood the radar found, against 1.4 % of quiet dry land. Only 12 % lie on permanent water or wetland; 78 % are on cropland.
* **That is not yet flood.** Cropland with open water may be flood the maps do not show, or fields flooded on purpose. So CSI 0.40–0.43 is an upper bound, not a score.
* **Most of the missed flood has no open water either.** Only 15 % of the mapped flood the radar misses is water for Sentinel-2. Water that receded or lay under vegetation fits both; this test cannot tell which.
* **One flood, one date.** This is EMSR664 only, with a Sentinel-2 pass about 29 hours after the radar.

*Data: Copernicus EMS Rapid Mapping, © European Union; Copernicus Sentinel-1 and Sentinel-2, processed by Sentinel Hub on the Copernicus Data Space Ecosystem; ESA WorldCover 2021, © ESA WorldCover project, CC BY 4.0.*